# Financial Agent Demo - Amazon Stock Analysis

This notebook demonstrates the deployed financial agent with:
- AWS Cognito authentication
- LangGraph agent with RAG
- Real-time and historical stock data
- Langfuse tracing
- Event streaming

## Setup and Configuration

In [ ]:
def authenticate_user(username: str, password: str):
    """
    Authenticate user with AWS Cognito and get ID token
    """
    client = boto3.client('cognito-idp', region_name=AWS_REGION)
    
    try:
        response = client.initiate_auth(
            ClientId=COGNITO_CLIENT_ID,
            AuthFlow='USER_PASSWORD_AUTH',
            AuthParameters={
                'USERNAME': username,
                'PASSWORD': password
            }
        )
        
        id_token = response['AuthenticationResult']['IdToken']
        access_token = response['AuthenticationResult']['AccessToken']
        
        print("Authentication successful!")
        return id_token, access_token
    
    except Exception as e:
        print(f"Authentication failed: {str(e)}")
        return None, None

username = input("Enter your Cognito username (email): ")
password = input("Enter your password: ")

id_token, access_token = authenticate_user(username, password)

if id_token:
    print("\nID Token (first 50 chars):", id_token[:50] + "...")

## AWS Cognito Authentication

In [ ]:
def query_agent(query: str, id_token: str):
    """
    Query the financial agent API with Cognito authentication
    """
    headers = {
        'Content-Type': 'application/json',
        'Authorization': id_token
    }
    
    payload = {
        'query': query
    }
    
    try:
        response = requests.post(API_ENDPOINT, headers=headers, json=payload)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {'error': str(e)}

def display_response(query: str, response: dict):
    """
    Display query and response in a formatted way
    """
    display(Markdown(f"### Query: {query}"))
    
    if 'error' in response:
        display(Markdown(f"**Error:** {response['error']}"))
    else:
        display(Markdown(f"**Response:**\n\n{response.get('response', 'No response')}"))
        display(Markdown(f"\n*Message count: {response.get('message_count', 0)}*"))
    
    print("\n" + "="*80 + "\n")

## Query Helper Function

In [ ]:
def query_agent(query: str, id_token: str):
    """
    Query the financial agent API with Cognito authentication
    """
    headers = {
        'Content-Type': 'application/json',
        'Authorization': id_token
    }
    
    payload = {
        'query': query
    }
    
    try:
        response = requests.post(API_ENDPOINT, headers=headers, json=payload)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {'error': str(e)}

def display_response(query: str, response: dict):
    """
    Display query and response in a formatted way
    """
    display(Markdown(f"### Query: {query}"))
    
    if 'error' in response:
        display(Markdown(f"**❌ Error:** {response['error']}"))
    else:
        display(Markdown(f"**Response:**\n\n{response.get('response', 'No response')}"))
        display(Markdown(f"\n*Message count: {response.get('message_count', 0)}*"))
    
    print("\n" + "="*80 + "\n")

## Test Query 1: Current Stock Price

In [ ]:
query1 = "What is the stock price for Amazon right now?"
response1 = query_agent(query1, id_token)
display_response(query1, response1)

## Test Query 2: Historical Stock Prices Q4

In [ ]:
query2 = "What were the stock prices for Amazon in Q4 last year?"
response2 = query_agent(query2, id_token)
display_response(query2, response2)

## Test Query 3: Compare Performance to Analyst Predictions

In [ ]:
query3 = "Compare Amazon's recent stock performance to what analysts predicted in their reports"
response3 = query_agent(query3, id_token)
display_response(query3, response3)

## Test Query 4: AI Business Research

In [ ]:
query4 = "I'm researching AMZN give me the current price and any relevant information about their AI business"
response4 = query_agent(query4, id_token)
display_response(query4, response4)

## Summary

This notebook demonstrated:

**Authentication**: Successfully authenticated with AWS Cognito user pool

**API Invocation**: Called the deployed Lambda function via API Gateway

**LangGraph Agent**: Agent processed queries using:
- Real-time stock price tool (yfinance)
- Historical stock price tool (yfinance)
- RAG search over Amazon earnings reports

**Streaming**: Events streamed via .astream() in the backend

**Langfuse Tracing**: All interactions traced and visible in Langfuse dashboard

**Infrastructure**: Deployed on AWS using Terraform:
- Lambda function
- API Gateway with Cognito authorizer
- S3 bucket for deployment
- Cognito User Pool

In [ ]:
query5 = "What is the total amount of office space Amazon owned in North America in 2024?"
response5 = query_agent(query5, id_token)
display_response(query5, response5)

## Langfuse Traces - API Response

After running the queries, we can fetch and display the Langfuse traces programmatically:

import requests
from datetime import datetime, timedelta

LANGFUSE_PUBLIC_KEY = os.getenv('LANGFUSE_PUBLIC_KEY')
LANGFUSE_SECRET_KEY = os.getenv('LANGFUSE_SECRET_KEY')
LANGFUSE_HOST = os.getenv('LANGFUSE_HOST', 'https://cloud.langfuse.com')

def get_langfuse_traces():
    """Fetch recent traces from Langfuse API"""
    url = f"{LANGFUSE_HOST}/api/public/traces"
    
    auth = (LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY)
    params = {
        'page': 1,
        'limit': 10
    }
    
    try:
        response = requests.get(url, auth=auth, params=params)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {'error': str(e)}

traces_data = get_langfuse_traces()

if 'error' not in traces_data:
    display(Markdown("### Langfuse Traces Retrieved Successfully"))
    display(Markdown(f"**Total traces found:** {traces_data.get('meta', {}).get('totalItems', 0)}"))
    
    traces = traces_data.get('data', [])
    if traces:
        display(Markdown("\n#### Recent Traces:"))
        for i, trace in enumerate(traces[:5], 1):
            display(Markdown(f"\n**Trace {i}:**"))
            display(Markdown(f"- **ID:** `{trace.get('id', 'N/A')}`"))
            display(Markdown(f"- **Name:** {trace.get('name', 'N/A')}"))
            display(Markdown(f"- **Timestamp:** {trace.get('timestamp', 'N/A')}"))
            display(Markdown(f"- **User ID:** {trace.get('userId', 'N/A')}"))
            
            if 'metadata' in trace:
                display(Markdown(f"- **Metadata:** {trace['metadata']}"))
    
    display(Markdown(f"\n**Full API Response:**"))
    print(json.dumps(traces_data, indent=2))
else:
    display(Markdown(f"**Error fetching traces:** {traces_data['error']}"))
    display(Markdown("\n**Alternative: View traces at:** https://cloud.langfuse.com"))

## Langfuse Trace Details

Fetch detailed information about a specific trace:

def get_trace_details(trace_id: str):
    """Fetch detailed trace information including observations"""
    url = f"{LANGFUSE_HOST}/api/public/traces/{trace_id}"
    auth = (LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY)
    
    try:
        response = requests.get(url, auth=auth)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {'error': str(e)}

if 'error' not in traces_data and traces_data.get('data'):
    first_trace_id = traces_data['data'][0].get('id')
    
    if first_trace_id:
        display(Markdown(f"### Detailed Trace Information for: `{first_trace_id}`"))
        
        trace_details = get_trace_details(first_trace_id)
        
        if 'error' not in trace_details:
            display(Markdown("\n#### Trace Overview:"))
            display(Markdown(f"- **Name:** {trace_details.get('name', 'N/A')}"))
            display(Markdown(f"- **User ID:** {trace_details.get('userId', 'N/A')}"))
            display(Markdown(f"- **Session ID:** {trace_details.get('sessionId', 'N/A')}"))
            display(Markdown(f"- **Timestamp:** {trace_details.get('timestamp', 'N/A')}"))
            
            observations = trace_details.get('observations', [])
            if observations:
                display(Markdown(f"\n#### Observations ({len(observations)} total):"))
                for obs in observations[:3]:
                    display(Markdown(f"\n**{obs.get('type', 'Unknown')}:**"))
                    display(Markdown(f"- **Name:** {obs.get('name', 'N/A')}"))
                    display(Markdown(f"- **Model:** {obs.get('model', 'N/A')}"))
                    
                    if 'usage' in obs:
                        usage = obs['usage']
                        display(Markdown(f"- **Token Usage:** Input: {usage.get('input', 0)}, Output: {usage.get('output', 0)}, Total: {usage.get('total', 0)}"))
                    
                    if 'latency' in obs:
                        display(Markdown(f"- **Latency:** {obs['latency']}ms"))
            
            display(Markdown("\n**Full Trace Details:**"))
            print(json.dumps(trace_details, indent=2))
        else:
            display(Markdown(f"**Error:** {trace_details['error']}"))
else:
    display(Markdown("**No traces available to display details.**"))

## Summary

This notebook demonstrated:

**Authentication**: Successfully authenticated with AWS Cognito user pool

**API Invocation**: Called the deployed Lambda function via API Gateway

**LangGraph Agent**: Agent processed queries using:
- Real-time stock price tool (yfinance)
- Historical stock price tool (yfinance)
- RAG search over Amazon earnings reports

**Streaming**: Events streamed via .astream() in the backend

**Langfuse Tracing**: All interactions traced and visible in Langfuse dashboard
- Traces fetched programmatically via Langfuse API
- Displayed trace IDs, metadata, observations, token usage, and latency
- Full API responses shown above

**Infrastructure**: Deployed on AWS using Terraform:
- Lambda function
- API Gateway with Cognito authorizer
- S3 bucket for deployment
- Cognito User Pool